# Hyperparameter Tuning

## Goals

The overarching goal is to learn to use callbacks for some typical tasks. These include:
- Reporting about training progress.
- Stoping once training no longer reduces loss.
- Tuning hyperparameters.
- Implementing adaptive learning rate decay.
- Finding an optimal batch-size for training.
- Putting some of this into ```my_keras_utils.py``` so that they can be easily called and reused.

## What's Here?

I continue working with MNIST data, which I began working with in [my first Keras models](first_model.ipynb). 

My **concrete objective** is to tune a model that does well on Kaggle: 97th percentile? That's tough, but I think I can make it work.

In [1]:
import numpy as np
from datetime import datetime, time, timedelta

import pandas as pd
import tensorflow as tf
import kerastuner as kt
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


import my_keras_utils as my_utils

In [2]:
tf.__version__
tf.config.experimental.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
## Load our data.
## Since the load process is a little slow, the try-except allows us to re-run all 
## cells without having to wait. 
try:
    ## Raises NameError and loads data if X_train is not defined.
    X_train.shape
except NameError: 
    ((X_train, y_train), (X_dev, y_dev), (X_test, y_test)) = my_utils.load_kaggle_mnist()




In [4]:

def overlay_histories(histories, metric):
    fig, ax = plt.subplots()
    n = 0
    for h in histories:
        x = range(0,len(h.history[metric]))
        y = np.array(h.history[metric])
        label = 'history_' + str(n)
        ax.plot(x,y, label=label)
        n += 1
    ax.legend()

## Tuning

Time to work with Keras Tuner.



In [5]:
def model_builder(hp):
    ## ### I need to learn about the options here. 'Choice' means "here are your choices"
    ## ### 'Int' is a different option that searches an integer range by steps.

    inputs = keras.Input(shape=(784))
    x = layers.experimental.preprocessing.Rescaling(1./255)(inputs)
    x = layers.Dropout(rate = hp.Float('drop_rate_1',
                                min_value = .10,
                                max_value = .45,
                                step = .02,
                                default = .25))(x)
    for i in range(hp.Int('num_layers', min_value = 2, max_value = 3, default = 2)):
        hp_units = hp.Int('units_l'+str(i+2), 
                            min_value = 80, 
                            max_value = 120, 
                            step = 20,
                            default = 100)
        x = layers.Dense(hp_units, activation = 'relu')(x)
        hp_dropout = hp.Float('drop_rate_'+str(i+2),
                                min_value = .10,
                                max_value = .45,
                                step = .01,
                                default = .15)
        x = layers.Dropout(rate=hp_dropout)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers
                                .Adam(hp.Float('learning_rate',
                                                min_value = .0003,
                                                max_value = .01,
                                                sampling = 'log',
                                                default = .001)),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

	


In [6]:
## Tuner callbacks
clear_output = my_utils.ClearTrainingOutput()
timed_update = my_utils.TimedProgressUpdate(3)
train_loss_stopping = keras.callbacks.EarlyStopping(monitor='loss', 
                            patience = 10, 
                            restore_best_weights = False
                            )

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 2,
                    cooldown= 2,
                    min_lr=.0003, 
                    factor=0.334,)

tuner_callbacks = [adaptive_lr, train_loss_stopping, clear_output]

In [7]:

hp = kt.HyperParameters()
hp.Int('num_layers', min_value = 2, max_value = 3, default = 2)
assert 'num_layers' in hp

test_tuner = kt.RandomSearch(model_builder,
                objective = 'val_loss',
                hyperparameters= hp,
                max_trials = 10,
                executions_per_trial = 2,
                tune_new_entries = False,
                directory = os.path.normpath('mnist'),
                project_name = 'test',
                overwrite = True)
                


test_tuner.search(X_train,
                y_train,
                validation_data = (X_dev, y_dev),
                epochs = 10,
                batch_size = 32,
                verbose = 1,
                callbacks = [])

Epoch 1/10
1188/1188 [==============================] - 3s 2ms/step - loss: 0.4447 - acc: 0.8623 - val_loss: 0.1949 - val_acc: 0.9425
Epoch 2/10
1188/1188 [==============================] - 3s 2ms/step - loss: 0.2259 - acc: 0.9302 - val_loss: 0.1492 - val_acc: 0.9610
Epoch 3/10
1188/1188 [==============================] - 3s 2ms/step - loss: 0.1818 - acc: 0.9431 - val_loss: 0.1253 - val_acc: 0.9635
Epoch 4/10
1125/1188 [===========================>..] - ETA: 0s - loss: 0.1574 - acc: 0.9503

KeyboardInterrupt: 

In [ ]:
import math
epochs_ = 70
factor_ = 2
total_epochs = epochs_*math.log(epochs_,factor_)**2
print(total_epochs/60*2)

In [ ]:
test_tuner.results_summary()

In [9]:
hp = kt.HyperParameters()
hp.Fixed('learning_rate', .003)

hyperband = kt.Hyperband(model_builder,
                     objective = 'val_loss', 
                     max_epochs = 70,
                     hyperparameters= hp,
                     hyperband_iterations = 2,
                     factor = 2,
                     directory = os.path.normpath('mnist'),
                     project_name = 'hb',
                     )	

hyperband.search_space_summary()

In [10]:
hyperband.search(X_train, y_train, 
            epochs=70,
            batch_size=32, 
            validation_data=(X_dev, y_dev),
            callbacks = tuner_callbacks,
            verbose = 0
            )


INFO:tensorflow:Oracle triggered exit


In [12]:
hyperband.results_summary()

## Ran the Hyperband, 

with the following params:
```
tuner = kt.Hyperband(model_builder,
                     objective = 'val_loss', 
                     max_epochs = 200,
                     hyperband_iterations = 10,
                     factor = 3,
                     directory = 'ignored/kt_trials',
                     project_name = 'dropout_mnist')	
###
tuner.search(X_train, y_train, 
            epochs=50,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = tuner_callbacks,
            )
```
Which involved way too many iterations. I didn't notice the hyperband_iterations param (10!).

### os.path.normpath

Per [Issue #198](https://github.com/keras-team/keras-tuner/issues/198) you may need to add os.path.normpath() to the directory keyword arg in Windows and the path to the logs needs to be short. (E.g., you won't be able to use my_trials/this_type_of_trial/this_trial--too long. Try mt/t/this).

In [ ]:
def rand_search_model_builder(hp):
    ## Define the hyperparameter search space.

    hp_dropout_x1 = hp.Float('rate1', min_value = .1, max_value = .4, step=.01)
    hp_dropout_w1 = hp.Float('rate2', min_value = .05, max_value = .4, step=.01)
    hp_dropout_w2 = hp.Float('rate3', min_value = .05, max_value = .4, step=.01)

    # Define the hypermodel
    inputs = keras.Input(shape=(784))
    x = layers.experimental.preprocessing.Rescaling(1./255)(inputs)
    x = layers.Dropout(rate = hp_dropout_x1)(x)
    x = layers.Dense(100, activation='relu',)(x)
    x = layers.Dropout(rate = hp_dropout_w1)(x)
    x = layers.Dense(100, activation='relu',)(x)
    x = layers.Dropout(rate = hp_dropout_w2)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

#kt.tuners.RandomSearch?
rand_tuner = kt.tuners.RandomSearch(rand_search_model_builder,
                     objective = 'val_loss', 
                     max_trials = 25,
                     executions_per_trial=1,
                     directory = os.path.normpath('ignored/mnist'),
                     project_name = 'rs')

rand_tuner.search_space_summary()

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 4,
                    cooldown= 8,
                    min_lr=.0003, 
                    factor=0.334,)

progress_update = my_utils.TimedProgressUpdate(2)


rand_tuner.search(X_train, y_train, 
            epochs=120,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        #clear_output
                        ],
            verbose = 0
            )

In [ ]:
def rand_search_model_builder_2(hp):
    ## Define the hyperparameter search space.

    hp_dropout_x1 = hp.Float('rate1', min_value = .27, max_value = .38, step=.001)
    hp_dropout_w1 = hp.Float('rate2', min_value = .05, max_value = .15, step=.001)
    hp_dropout_w2 = hp.Float('rate3', min_value = .11, max_value = .21, step=.001)

    # Define the hypermodel
    inputs = keras.Input(shape=(784))
    x = layers.experimental.preprocessing.Rescaling(1./255)(inputs)
    x = layers.Dropout(rate = hp_dropout_x1)(x)
    x = layers.Dense(100, activation='relu',)(x)
    x = layers.Dropout(rate = hp_dropout_w1)(x)
    x = layers.Dense(100, activation='relu',)(x)
    x = layers.Dropout(rate = hp_dropout_w2)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )

    return model

#kt.tuners.RandomSearch?
new_tuner = kt.tuners.RandomSearch(rand_search_model_builder_2,
                     objective = 'val_loss', 
                     max_trials = 25,
                     executions_per_trial=1,
                     directory = os.path.normpath('ignored/mnist'),
                     project_name = 'rs_2')

new_tuner.search_space_summary()

adaptive_lr = keras.callbacks.ReduceLROnPlateau(
                    monitor='loss',
                    patience= 4,
                    cooldown= 8,
                    min_lr=.0003, 
                    factor=0.334,)

progress_update = my_utils.TimedProgressUpdate(.5)

new_tuner.search(X_train, y_train, 
            epochs=1,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        #clear_output
                        ],
            verbose = 0
            )

### A dubious fist run

I ran a random_search, lr = .001 and a 0-.5 range on the params; 50 epochs, 2 executions per trial. The results were okay, but it seemed like the number of epochs was too few. Also, the max_trials was too big. At least on the Surface, it took three minutes per trial. Ouch. 5 hours later...

### I ran a second time

This a second time. This time with more epochs (120), but 1 execution per trial and a slightly different band of paramters. There was a clear stand out

Trial ID: 39f9768e653c7179ddb0917bd84a4b9d
|-Score: 0.05155862867832184
|-Best step: 0
Hyperparameters:
|-rate1: 0.32999999999999985
|-rate2: 0.1
|-rate3: 0.16000000000000003

In [ ]:
new_tuner.results_summary()

## Baysian Tuning

Let's see how well the Baysian Optimizer does with the rand_search_builder.

In [ ]:
bayesian_tuner = kt.tuners.BayesianOptimization(
            rand_search_model_builder,
            objective='val_loss',
            max_trials = 25,
            directory = os.path.normpath('ignored/mnist'),
            project_name = 'bayes'
)

bayesian_tuner.search(X_train, y_train, 
            epochs=120,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 0
            )


In [ ]:
bayesian_tuner.results_summary()

## Let's put it all together. 

The Bayesian tuner got pretty bad results compared with the other two. I'm not really sure what happened, except that it didn't seem exploratory enough. I should probably figure out how to get better Bayesian results.

But for now, we have several high scoring models. Let's extract three and cross validate.  

In [ ]:
best = rand_tuner.get_best_models()[0]
config = best.get_config()
model_1 = best.from_config(config)
config

In [ ]:
temp = rand_tuner.get_best_models(num_models=2)
model_2 = temp[0].from_config(temp[0].get_config())
model_3 = temp[1].from_config(temp[1].get_config())


In [ ]:
def cross_validate(model, X, y, val_size = 2000, folds = 3):
    init_weights = model.get_weights()
    results = []
    for fold in range(folds):
        X_val = X[fold*val_size:(fold+1)*val_size,:]
        X_train = np.concatenate((X[0:fold*val_size,:],X[(fold+1)*val_size:,:]))
        y_val = y[fold*val_size:(fold+1)*val_size]
        y_train = np.concatenate((y[0:fold*val_size],y[(fold+1)*val_size:]))
        history = model.fit(X_train, y_train,
            epochs=200,
            batch_size=128, 
            validation_data=(X_val, y_val),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 0)
        model.set_weights(init_weights)
        results.append(history)
    return results

In [ ]:
results = []
for model in [model_1, model_2, model_3]:
    model.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )
    histories = cross_validate(model, X_train, y_train)
    results.append(histories)

In [ ]:
for e in results[0]:
    print(e.history['val_loss'][-1])
for e in results[1]:
    print(e.history['val_loss'][-1])
for e in results[2]:
    print(e.history['val_loss'][-1])
    

In [ ]:
## Reset all the weights.
model_1.compile(optimizer = keras.optimizers.Adam(.003),
                    loss = "sparse_categorical_crossentropy",
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
    )
## Save so I can move to another computer
model_1.save('MNIST_model.hdf5')
model_2.save("mnist_2.hdf5")
model_3.save("mnist_3.hdf5")

In [ ]:
model_1.fit(X_train, y_train,
            epochs=200,
            batch_size=128, 
            validation_data=(X_dev, y_dev),
            callbacks = [adaptive_lr, 
                        progress_update, 
                        clear_output
                        ],
            verbose = 2)
model_1.save('MNIST_model.hdf5')